# CivicPulse · Anonymize the pothole pair photos with SAM 3.1 (open weights, free Colab GPU)

Every photo is anonymized **here, before any Meta Model API call**: the free API credits may be
used by Meta for product improvement, so the API only ever sees these anonymized copies.

| Step | What |
|---|---|
| 1 | T4 GPU, install the official `sam3` package (pinned), Hugging Face login (access to `facebook/sam3.1`) |
| 2 | Load SAM 3.1 |
| 3 | Unzip `needs_sam.zip` from your Drive (made by `data/pothole_pairs/prefilter.py --sam`: only the photos where a local detector saw a face, a person or a vehicle) |
| 4 | Per photo: EXIF orientation applied, resize to 1600 px, SAM 3.1 masks for **person** and **license plate** (+ OpenCV face detector as a safety net), heavy blur + pixelation inside the dilated masks, new JPEG from pixels only (no metadata) |
| 5 | Contact sheet for manual review + `report.csv` |
| 6 | `anon.zip` back to your Drive → unzip into `data/private/anon/` |

**Privacy choices (max anonymization, decided by sebas 2026-10-27):** whole people are blurred, not only
faces (crew members appear in after photos); low thresholds so doubtful detections are blurred too
(over-blurring is fine, potholes are on the road surface). Resumable: photos already in the output
folder on Drive are skipped, so a disconnected Colab session continues where it stopped.

Optional step: by default `prefilter.py` blurs on your CPU. With `--sam` it sends only the flagged photos here for finer masks.
Time: about 2 prompts × ~4 s per photo on a T4 → ~1,000 photos per session of ~2.5 h. Run it in
several sessions if needed. Falls back to `raw.zip` (every photo) when `needs_sam.zip` is not on Drive.

## 1. GPU, install, login

In [ ]:
import sys, torch
assert torch.cuda.is_available(), "Select a GPU: Runtime → Change runtime type → T4 GPU"
props = torch.cuda.get_device_properties(0)
print(f"GPU: {props.name} | VRAM {props.total_memory / 1024**3:.1f} GB | Python {sys.version.split()[0]}")

In [ ]:
import subprocess

SAM3_COMMIT = "7f473cfd983d5eb43040ff608f66bff0d7bdcb53"

def pip(*args):
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *args])

# --no-deps: the repo pins numpy<2, which would break Colab's OpenCV. Real deps installed below.
pip("--no-deps", f"git+https://github.com/facebookresearch/sam3.git@{SAM3_COMMIT}")
pip("timm>=1.0.17", "ftfy==6.1.1", "regex", "iopath>=0.1.10", "typing_extensions", "huggingface_hub",
    "tqdm", "psutil", "pycocotools", "einops", "scipy", "setuptools<81", "opencv-python-headless<5", "pillow")
print("installed")

In [ ]:
# Needs a Hugging Face token with access to facebook/sam3.1, stored as the Colab secret HF_TOKEN.
from google.colab import userdata
from huggingface_hub import login
login(token=userdata.get("HF_TOKEN"))

## 2. Load SAM 3.1

In [ ]:
import functools, gc, inspect, time
from sam3.model_builder import build_sam3_multiplex_video_predictor, download_ckpt_from_hf

ckpt = download_ckpt_from_hf(version="sam3.1")
predictor = build_sam3_multiplex_video_predictor(
    checkpoint_path=ckpt, use_fa3=False, compile=False, warm_up=False, async_loading_frames=False)

# Official start_session passes kwargs that the multiplex init_state rejects (sam3 @7f473cf).
model = predictor.model
original = model.init_state
params = inspect.signature(original).parameters
if not any(p.kind == inspect.Parameter.VAR_KEYWORD for p in params.values()):
    @functools.wraps(original)
    def init_state_compatible(*args, **kwargs):
        return original(*args, **{k: v for k, v in kwargs.items() if k in params})
    object.__setattr__(model, "init_state", init_state_compatible)
print("SAM 3.1 ready:", ckpt)

## 3. Input and output on your Drive

In [ ]:
from pathlib import Path
import zipfile
from google.colab import drive
drive.mount("/content/drive")

DRIVE = Path("/content/drive/MyDrive/civicpulse")
RAW_ZIP = DRIVE / "needs_sam.zip"                                    # upload data/private/needs_sam.zip here
if not RAW_ZIP.exists():
    RAW_ZIP = DRIVE / "raw.zip"                                      # no prefilter: every photo
print("input:", RAW_ZIP.name)
OUT_DIR = Path("/content/drive/MyDrive/civicpulse/anon")             # anonymized photos, resumable
RAW_DIR = Path("/content/raw")
OUT_DIR.mkdir(parents=True, exist_ok=True)
if not RAW_DIR.exists():
    with zipfile.ZipFile(RAW_ZIP) as zf:
        zf.extractall(RAW_DIR)
photos = sorted(p for p in RAW_DIR.iterdir() if p.suffix.lower() in {".jpg", ".jpeg", ".png", ".webp"})
todo = [p for p in photos if not (OUT_DIR / f"{p.stem}.jpg").exists()]
print(f"{len(photos)} photos, {len(todo)} still to anonymize")

## 4. Anonymize

In [ ]:
import csv, io
import cv2
import numpy as np
from PIL import Image, ImageOps

PROMPTS = ["person", "license plate"]
SAM_THRESH = 0.25          # low on purpose: a doubtful detection is blurred
MAX_SIDE = 1600
WORK = Path("/content/work"); WORK.mkdir(exist_ok=True)
face_cascade = cv2.CascadeClassifier(cv2.data.haarcascades + "haarcascade_frontalface_default.xml")


def sam_masks(image_path, h, w):
    masks, counts = [], {}
    sid = predictor.handle_request(dict(type="start_session", resource_path=str(image_path)))["session_id"]
    try:
        for prompt in PROMPTS:
            predictor.handle_request(dict(type="reset_session", session_id=sid))
            out = predictor.handle_request(dict(type="add_prompt", session_id=sid, frame_index=0,
                                                text=prompt, output_prob_thresh=SAM_THRESH))["outputs"]
            m = out.get("out_binary_masks")
            m = (m.detach().cpu().numpy() if hasattr(m, "detach") else np.asarray(m)) if m is not None else np.zeros((0, h, w))
            m = m.astype(bool).reshape(-1, *m.shape[-2:]) if m.size else np.zeros((0, h, w), bool)
            kept = [cv2.resize(x.astype(np.uint8), (w, h), interpolation=cv2.INTER_NEAREST).astype(bool) for x in m if x.any()]
            counts[prompt] = len(kept)
            masks += kept
    finally:
        predictor.handle_request(dict(type="close_session", session_id=sid))
        gc.collect(); torch.cuda.empty_cache()
    return masks, counts


def blur_region(rgb, mask):
    h, w = rgb.shape[:2]
    k = max(15, int(0.02 * max(h, w))) | 1
    grown = cv2.dilate(mask.astype(np.uint8), cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (k, k))).astype(bool)
    small = cv2.resize(rgb, (max(1, w // 24), max(1, h // 24)), interpolation=cv2.INTER_LINEAR)
    pixelated = cv2.resize(small, (w, h), interpolation=cv2.INTER_NEAREST)
    hidden = cv2.GaussianBlur(pixelated, (0, 0), sigmaX=max(h, w) / 80)
    rgb[grown] = hidden[grown]
    return rgb


def anonymize(src):
    img = ImageOps.exif_transpose(Image.open(src)).convert("RGB")
    img.thumbnail((MAX_SIDE, MAX_SIDE))
    rgb = np.array(img)
    h, w = rgb.shape[:2]
    tmp = WORK / f"{src.stem}.jpg"
    Image.fromarray(rgb).save(tmp, quality=95)
    masks, counts = sam_masks(tmp, h, w)
    faces = face_cascade.detectMultiScale(cv2.cvtColor(rgb, cv2.COLOR_RGB2GRAY), 1.1, 4, minSize=(20, 20))
    for x, y, fw, fh in faces:
        m = np.zeros((h, w), bool); m[y:y + fh, x:x + fw] = True
        masks.append(m)
    counts["face_cascade"] = len(faces)
    if masks:
        rgb = blur_region(rgb, np.logical_or.reduce(masks))
    Image.fromarray(rgb).save(OUT_DIR / f"{src.stem}.jpg", "JPEG", quality=88)  # pixels only, no EXIF
    tmp.unlink(missing_ok=True)
    return counts


report_path = OUT_DIR / "report.csv"
new_file = not report_path.exists()
with report_path.open("a", newline="") as f:
    writer = csv.writer(f)
    if new_file:
        writer.writerow(["photo", "person", "license_plate", "face_cascade", "seconds", "reviewed"])
    t_start = time.time()
    for i, src in enumerate(todo, 1):
        t = time.time()
        c = anonymize(src)
        writer.writerow([src.stem, c["person"], c["license plate"], c["face_cascade"], round(time.time() - t, 1), ""])
        if i % 25 == 0:
            f.flush()
            rate = (time.time() - t_start) / i
            print(f"{i}/{len(todo)} · {rate:.1f} s/photo · ~{rate * (len(todo) - i) / 60:.0f} min left")
print("done")

## 5. Manual review (contact sheet)
Look for any face, person or plate that is still readable. Mark the photo names in `report.csv` (`reviewed` column) and fix them by hand or drop them.

In [ ]:
import random
import matplotlib.pyplot as plt
done = sorted(OUT_DIR.glob("*.jpg"))
sample = random.Random(0).sample(done, min(24, len(done)))
fig, axs = plt.subplots(4, 6, figsize=(24, 16))
for ax, p in zip(axs.flat, sample):
    ax.imshow(Image.open(p)); ax.set_title(p.stem, fontsize=8); ax.axis("off")
plt.tight_layout(); plt.show()

## 6. Pack for your computer
Download `anon.zip` from Drive and unzip it into `data/private/anon/` of the repo.

In [ ]:
with zipfile.ZipFile(OUT_DIR.parent / "anon.zip", "w", zipfile.ZIP_STORED) as zf:
    for p in sorted(OUT_DIR.iterdir()):
        zf.write(p, p.name)
print("anon.zip ->", OUT_DIR.parent / "anon.zip")